# Step 1: Create master catalog of sources for cutouts generation
This step of the RRN-to-Zooniverse pipeline will search for calibrated L3 and L4 data products that match an input program, plan, pass, and segment number, along with input filters. Then the files are parsed to build a master catalog of source positions, AWS S3 location to the filter-specific images the source was detected in, and source metadata for later filtering (magntitudes, sizes, segment areas, etc.).



In [ ]:
import s3fs #To access cloud files as though they were local
from pprint import pprint
import asdf #To handle ASDF input and output
import pyarrow.parquet as pq

import numpy as np

from astropy.coordinates import SkyCoord, match_coordinates_sky
from astropy import units as u

import os

import pandas as pd

from datetime import datetime

from tqdm import trange

In [ ]:
def Retrieve_file_matches(dir_uri, filter_match, All_files=True, in_plan=None, in_pass=None, in_segment=None):
    '''
    
    Retrieves a list of the files stored on AWS that match 
    the input dir_uri root directory, and input filter, plan, 
    pass, and segment. Also splits off separate arrays that 
    contain the visit-level info and sky-cell info associated 
    with each returned file. If All_files = True, returns all 
    files that match the input filter. Otherwise, an input 
    plan, pass, and segment 
    are also required.

    Parameters:
    -----------
    dir_uri: str
        the input directory to parse
    filter_match: str
        the filter to match against
    All_files: boolean
        Whether to return all files that match the input filter
        or impose filter on plan, pass, and segment as well
    in_plan: str
        The plan number. A specification in APT of the Pass Plans  
        comprising a Program. Should have format "01" for plan 1
    in_pass: str
        The pass number. The collection of activities generated from 
        each iteration of a Pass Plan. Multiple Passes may be generated 
        from the same Pass Plan to allow repetition or execute at 
        different orientations (via special requirements).
    in_segment: str
        The sequence of activities produced by each iteration of the 
        Observation Sequences defined in a Segment Plan. A Segment 
        may include multiple traversals of Mosaic Pattern(s), with 
        filter wheel moves occurring between Observations. The 
        Segment will be the unit of planning, while short term 
        scheduling will be done at the Visit level.
    
    Returns:
    --------
    visit_level_info_array: nd.array
        array of the viti-level combo information for each matched 
        program file.
    skycell_info_array: nd.array
        the skycell information for each matched program file
    prog_match_files: list
        the list of program files matched to the input filter, plan, 
        pass, and segment 
    '''
    file_list = fs.ls(dir_uri)
    print(f'Full length of file list available at {dir_uri} = ', len(file_list))
    
    print(f'Looking for files taken in filter {filter_match}')
    
    prog_match_files = []
    plan_visit_level_info = []
    visit_info = []
    skycell_info = []
    filter_info = []
    suffix = []
    for i in range(len(file_list)):
        # print('s3://'+file_list[i])
        if check_if_uploaded('s3://'+file_list[i]):
            # print(check_if_uploaded('s3://'+file_list[i]))
            # print(file_list[i])
            continue
        else:
            suffix_inf = file_list[i].rsplit('/')[-1].rsplit('_')[-1].rsplit('.')[0]
            visitlev_inf = file_list[i].rsplit('/')[-1].rsplit('_')[-4]
            filter_inf = file_list[i].rsplit('/')[-1].rsplit('_')[-2]
            skycell_inf = file_list[i].rsplit('/')[-1].rsplit('_')[-3]
            # print('visit-level-info = ', visitlev_inf)
            plan_current = visitlev_inf[1:3]
            pass_current = visitlev_inf[3:6]
            segment_current = visitlev_inf[6:9]
            visit_current = visitlev_inf[12:15]
            # print(plan_current, pass_current, segment_current)
            
            
            # filter_info.append(filter_inf)
            if All_files == True:
                if filter_inf == filter_match and suffix_inf == 'coadd':
                    
                    plan_visit_level_info.append(visitlev_inf)
                    visit_info.append(visit_current)
                    skycell_info.append(skycell_inf)
                    prog_match_files.append(file_list[i])
            else:
                if filter_inf == filter_match and suffix_inf == 'coadd' and plan_current== in_plan and pass_current == in_pass and segment_current == in_segment:
                    plan_visit_level_info.append(visitlev_inf)
                    visit_info.append(visit_current)
                    skycell_info.append(skycell_inf)
                    prog_match_files.append(file_list[i])
    
    print(f'Summary of files:')        
    if len(plan_visit_level_info) > 0:
        len_fl = len(plan_visit_level_info)
        print(f'{len_fl} files found that match input filter request!')
        # pprint(visit_level_info)
    else:
        print('No files found!')

    plan_visit_level_info_array = np.array(plan_visit_level_info)
    skycell_info_array = np.array(skycell_info)
        
    return plan_visit_level_info_array, skycell_info_array, np.array(prog_match_files), np.array(visit_info)

def Print_file_info(visit_level_info_array, skycell_info_array):
    '''
    Given an input array of visit-level combinations and skycells, 
    print a summary of the visit-level and skycell information.  

    Prameters:
    ---------
    visit_level_info_array: nd.array
        an array of visit-level descriptors that summarize the 
        plan, pass, segment, observations and visit of each file
    
    skycell_info_array: nd.array
        an array of the skycell info associated with each 
        visit-level descriptor of the input program/filter

    Returns:
    --------
    None
        
    '''

    print('Number of unique visit-level combos :', len(np.unique(visit_level_info_array)))
    print('Number of unique skycell positions:', len(np.unique(skycell_info_array)))
    unique_visit_level_info = np.unique(visit_level_info_array)
    

    PASS_info = []
    SEG_info = []
    OBS_info = []
    VISIT_info = []
    SKYCELL_info = []
    
    
    for i in range(len(unique_visit_level_info)):
        VL_info = unique_visit_level_info[i]
        # these are hard coded to match the format of visit-level info of Roman WFI files
        # https://roman-docs.stsci.edu/data-handbook/data-levels-and-products
        pass_info = VL_info[3:6]
        seg_info = VL_info[6:9]
        obs_info = VL_info[9:12]
        visit_info = VL_info[12:]

        PASS_info.append(pass_info)
        SEG_info.append(seg_info)
        OBS_info.append(obs_info)
        VISIT_info.append(visit_info)
    
        visit_match = np.where(visit_level_info_array==VL_info)[0]
        skycell_match = skycell_info_array[visit_match]

        
    unique_PASS_info = np.unique(PASS_info)
    num_passes = len(unique_PASS_info)
    print(f'Found {num_passes} unique passes: ', unique_PASS_info)
    
    unique_SEF_info = np.unique(SEG_info)
    num_segs = len(unique_SEF_info)
    print(f'Found {num_segs} unique segments', unique_SEF_info)
    
    unique_OBS_info = np.unique(OBS_info)
    num_obs = len(unique_OBS_info)
    print(f'Found {num_obs} unique observations', unique_OBS_info)
    
    PASS_info_array = np.array(PASS_info)
    SEG_info_array = np.array(SEG_info)
    OBS_info_array = np.array(OBS_info)
    VISIT_info_array = np.array(VISIT_info)
    
    print('======================================')
    for pid in unique_PASS_info:

        id_match = np.where(PASS_info_array == pid)
        seg_match = SEG_info_array[id_match]
        obs_match = OBS_info_array[id_match]
        visit_match = VISIT_info_array[id_match]
    
        print(f'Pass {pid} has...')
        print(f'{len(np.unique(seg_match))} segment: {np.unique(seg_match)}')
        print(f'{len(np.unique(obs_match))} observations: {np.unique(obs_match)}')
        print(f'{len(np.unique(visit_match))} visits: {np.unique(visit_match)}')
        print('======================================')

def update_txt_file(file):
    """Appends a new L3 file name to the prog_{program_num}-L3-file-select-tracking.txt file."""
    with open(filename, 'a') as f:
        f.write(str(file) + '\n')

        
def get_uploaded_ids(filename):
    with open(filename, 'r') as f:
        processed_files = [line.strip() for line in f]
    return np.array(processed_files)


def check_if_uploaded(file):
    """Checks if the given id_field is already in uploaded_subjects_by_id_field.txt."""
    try:
        # with open(filename, 'r') as f:
        return str(file) in processed_files
    except FileNotFoundError:
        print('Tracking file does not exist!')
        return False  # If the file doesn't exist yet, nothing has been uploaded

# Summary of file name format for L3 and L4 data products:
[Read more here](https://roman-docs.stsci.edu/data-handbook/wfi-data-levels-and-products#WFIDataLevelsandProducts-Level3and4FileNameslevel34-Filenames)
- an example of the file naming structure: "r00342_p_v0100100100100_008m46x51y68_f087"

- the first part, "r00342," is the program

- "p" indicates that this is a prompt data product (not an officiel data release product)

- each file has a visit-level descriptor known as a "subset": "v01002001001001," which can be broken into components "v01|002|001|001|001"
    - This is a visit-level combination of co-added (dithered) exposures from execution plan 01, pass 002, segment 001, observation 001, visit 001. 

- for the [SOC HLWAS simulation](https://outerspace.stsci.edu/pages/viewpage.action?pageId=389808641&spaceKey=WFISIMWG&title=SOC%2BSimulation%2BProducts), pass 013 corresponds to the medium tier HLWAS 


- for each filter, this pass (013) should have plan 01, segment 001, and visits 001-018 (one for each tile of the HLWAS mosaic)


- for each filter, the "observation" descriptor varies: f106: 001; f129: 002; f158: 003


- Need to indentify all files that correspond to plan 01, pass 013, and segment 001


- the second-to-last part of the file naming structure is the skycell. The skycell information "008m46x51y68" contains the center of the projection region (008m46 = RA,DEC = 08, -46) and the center of the sky cell within the proection region in pixel coords (x51y68 = x,y = 51,68)


In [ ]:
'''
find out how many unique visit-level combinations for each filter to help get an idea of what "observations" this dataset includes
'''

file_source = 'soc_simulations'

# the program, plan, pass, and segment to be searched
program_num = 'r00342'
in_plan_ = '01'
in_pass_ = '013'
in_segment_ = '001'

# the root directory of the program
dir_uri = f's3://stpubdata/roman/nexus/soc_simulations/{program_num}/'

# we need filters YJH
in_filter = 'f106' #primary filter
f1 = 'f129' #secondary filter
f2 = 'f158' #tertiary filter
bands = [in_filter, f1, f2]

# base name of the master catalog
catalog_name = '10122026-TEST-full_catalog_file_locs_'
for i in range(len(bands)):
    if bands[i] == bands[-1]:
        catalog_name = catalog_name + bands[i]
    else:
        catalog_name = catalog_name + bands[i]+'_'
print(catalog_name)


#connects us to AWS S3 bucket where the data is stored; anon=True means we only want access to public data files
fs = s3fs.S3FileSystem(anon=True) 


In [ ]:
# For tracking which L3 images on AWS have already been added to the master source catalog
# Create the file initially (if it doesn't exist)
filename = f'/teams/gz-roman/10122026-TEST-prog_{program_num}-filter_{bands[0]}-L3-file-select-tracking.txt'
print(filename)

# Make sure the file exists
open(filename, 'a').close()

# check which files have already been "scraped" for sources
processed_files = get_uploaded_ids(filename)
print('Files that have already been checked for L4 catalog sources: ')
print(len(processed_files))
print(processed_files)


In [ ]:
# define empty dictionary to hold file match info for each filter
band_info = {}

for i in trange(len(bands)):
    print(f'=========================================={bands[i]}==========================================')
    # retrieve list of files that match input program, plan, pass, and segment
    filter_plan_visit_level_info_array, filter_skycell_info_array, filter_prog_match_files, filter_visit_info = Retrieve_file_matches(dir_uri+'l3/', bands[i], All_files=False, in_plan=in_plan_, in_pass=in_pass_, in_segment=in_segment_)

    # print a summary of the file list
    Print_file_info(filter_plan_visit_level_info_array, filter_skycell_info_array)
    
    band_info[bands[i]] = {'plan_visit_level_info': filter_plan_visit_level_info_array, 
                      'skycell_info_array':filter_skycell_info_array, 
                      'prog_match_files':filter_prog_match_files,
                      'visit_info':filter_visit_info}



# Define the columns to be included in the master source catalog
Can check the source detection [documentation](https://roman-pipeline.readthedocs.io/en/stable/roman/source_catalog/main.html#source-detection) or print the available columns using the next celll 

In [ ]:
# check available catalog columns using the first matched file

catalog_file = dir_uri+'l4/l3_source_catalog/'+band_info[bands[0]]['prog_match_files'][0].rsplit('/')[-1].rstrip('_coadd.asdf')+'_cat.parquet'
print(f'Catalog file: {catalog_file} \n')
parquet_file = pq.read_table(catalog_file, filesystem=fs)
print('Catalog column names: \n')
print(parquet_file.schema.names)

In [ ]:
do_not_copy = ['asdf_loc', 'id_str']

# check if a master catalog with the name defined above exists, if so, use that.
# this step helps resume the creation of the master catalog if it was only 
# partially created (e.g, due to an interuption or to manual batch creation). 
if os.path.isfile('/teams/gz-roman/'+catalog_name+'.parquet'):
    print(f'Catalog for band combination {bands} already exists!')
    print('Reading in existing catalog!')
    catalog_df = pd.read_parquet('/teams/gz-roman/'+catalog_name+'.parquet')
    catalog_columns = catalog_df.columns

    catalog_keys_temp = [catalog_columns[i].rsplit('_f')[0] for i in range(len(catalog_columns))]
    catalog_keys, indices = np.unique(catalog_keys_temp, return_inverse=True)

    indices1 = indices[:2]
    indices2 = indices[len(bands)+1:len(catalog_keys)+len(bands)-1]
    indices_corrected = np.concatenate((indices1, indices2))
    catalog_keys = catalog_keys[indices_corrected]
    catalog_keys = [catalog_keys[i] for i in range(len(catalog_keys)) if catalog_keys[i] not in do_not_copy]

else:
    print(f'Creating new catalog for band combination {bands}!')
    
    catalog_columns = ['id_str']
    catalog_keys = []
    for i in range(len(bands)):
        catalog_columns.append(f'asdf_loc_{bands[i]}')

    # if creating a new catalog, define the column names to include from individual L4 data products
    catalog_keys.extend(['ra_centroid', 'dec_centroid'])
    catalog_keys.extend(['kron_abmag', 'kron_radius', 'fluxfrac_radius_50', 'segment_area', 'is_extended', 
                        'semimajor', 'semiminor'])
    
    
    for band in bands:
        catalog_columns.extend([catalog_keys[i]+f'_{band}' for i in range(len(catalog_keys))])
        
    
    catalog_df = pd.DataFrame(columns=catalog_columns)
    
# print(catalog_df)
# print(catalog_columns)
# print(catalog_keys)

# for sources which don't have an match filter file at a given primary filter skycell
# at a later step, these missing files can be rechecked for and added to the existing catalog
missing_key_fill_value = [-999.0, -999.0, -999.0, -999.0, -999.0, -999.0, 'MISSING', -999.0, -999.0] 

# Start from the list of subsets (visit-level combinations) returned for the primary filter.
- for every program file that matches the in_filter and pass_number (which is an observation + visit + skycell combination), identify the file name for the match_filters that overlap with the skycell

*Note: the matched files in each filter may contain up to 4 visits (single pointings) that overlap at the given skycell. In the simulated HLWAS survey, the visit numbers should be adjacent to each other in the mosaic (e.g., 1,2,11,12 or 2,3,10,11) becuase the simulated moasic proceeds in an S-pattern of 18 visits - see the [simulation summary](https://outerspace.stsci.edu/pages/viewpage.action?pageId=389808641&spaceKey=WFISIMWG&title=SOC%2BSimulation%2BProducts) for more details. 


In [ ]:
# # FWHM (miliarcsec) for f106, f129, and f158 = 87, 106, 128 https://roman-docs.stsci.edu/roman-instruments/the-wide-field-instrument/observing-with-the-wfi/wfi-quick-reference#WFIQuickReference-Table-WFI-Filters
# FWHM = np.mean([87, 106, 128])/(1e3)
# PSF_area = 1.133*pow(FWHM,2)
# print(PSF_area, 'arcsec^2')
# src_id = 0


In [ ]:
ims_wo_sources = 0 # track how many image files are "empty"
skycell_match_only = 0 #track how many skycells that exist for the primary filter had a skycell match but not a visit (mosaic tile) match
max_sep = (0.5/3600) * u.degree # for crossmatching source positions between source in primary filter image and matching filter images
num_band_files_per_image = [] # for tracking the number of primary filter files for which matching filter files were available

startTime = datetime.now()

for i in trange(len(band_info[bands[0]]['prog_match_files'])):
    band_files = []
    band_catalogs = []

    # for each primary filter file, find the matched filter files that have the same skycell
    for b in bands:
        if b == bands[0]:
            # =============================================================================================
            # define primary filter image and catalog URIs for the current visit- and skycell-level descriptor
            file_name_coadd = 's3://'+band_info[bands[0]]['prog_match_files'][i]
            file_name_cat = dir_uri+'l4/l3_source_catalog/'+file_name_coadd.rsplit('/')[-1].rstrip('_coadd.asdf')+'_cat.parquet'
            skycell = band_info[bands[0]]['skycell_info_array'][i]
            
        else:
            match_i = np.where((band_info[b]['skycell_info_array']==band_info[bands[0]]['skycell_info_array'][i]) & (band_info[b]['visit_info']==band_info[bands[0]]['visit_info'][i]))[0]
            len_match = len(np.array(band_info[b]['prog_match_files'])[match_i])

            if len_match < 1:
                print(f'Could not find visit AND skycell match for band {b}. Trying skycell match only!')
                match_i = np.where((band_info[b]['skycell_info_array']==band_info[bands[0]]['skycell_info_array'][i]))[0]
                len_match = len(match_i)
                
                if len_match < 1:
                    print('Filters matching the current skycell do not exist! Adding MISSING flag to catalog')
                    file_name_coadd = 'MISSING'
                    file_name_cat = 'MISSING'
                    
                else:
                    skycell_match_only = skycell_match_only + 1
                    file_name_coadd = 's3://'+np.array(band_info[b]['prog_match_files'])[match_i][0] #take the first match in the list of retunred skycell matches
                    file_name_cat = dir_uri+'l4/l3_source_catalog/'+file_name_coadd.rsplit('/')[-1].rstrip('_coadd.asdf')+'_cat.parquet'
           
            else:
                file_name_coadd = 's3://'+np.array(band_info[b]['prog_match_files'])[match_i][0] #take the first match in the list of retunred skycell matches
                file_name_cat = dir_uri+'l4/l3_source_catalog/'+file_name_coadd.rsplit('/')[-1].rstrip('_coadd.asdf')+'_cat.parquet'
            
        band_files.append(file_name_coadd)
        band_catalogs.append(file_name_cat)


            
    # #  ===================================================================================
    # read in associated catalog file:
    # Read the file metadata schema 
    # this is to get the number of sources detected in the image
    parquet_file = pq.read_table(band_catalogs[0], filesystem=fs)
    columns = parquet_file.schema.names
    cat_table = parquet_file.to_pandas()
    num_src_in_image = len(cat_table)

    # track number of images without sources
    if num_src_in_image == 0:
        ims_wo_sources = ims_wo_sources + 1
        continue

    #empty subtable (DF) to populate for current prog_match_file
    append_ = pd.DataFrame(columns=catalog_columns)
    
    if len(catalog_df['id_str'].values) == 0:
        #if no sources in the catalog yet, start source ID indexing at 0
        append_['id_str'] = np.arange(0,num_src_in_image)
    else:
        # else, check last appended source and start source ID indexing at last src_id + 1
        append_['id_str'] = np.arange(catalog_df['id_str'].values[-1]+1,catalog_df['id_str'].values[-1]+1+num_src_in_image)

    
    for b in range(len(bands)):
        band = bands[b]

        # for each band, append the asdf file location for all sources in that band image
        append_[f'asdf_loc_{band}'] = num_src_in_image*[band_files[b]]

        
        if band == bands[0]:
            # use the first filter to match source locations
            parquet_file = pq.read_table(band_catalogs[b], filesystem=fs)
            columns = parquet_file.schema.names
            cat_table0 = parquet_file.to_pandas()

            # primary filter source locations converted into astropy coordinates
            RA0, DEC0 = cat_table0['ra_centroid'], cat_table0['dec_centroid']
            c0 = SkyCoord(RA0.to_numpy()*u.deg, DEC0.to_numpy()*u.deg, frame='icrs')

            # for each catalog_key, append the source info from the catalog to the information subtable append_
            for c in range(len(catalog_keys)):
                if catalog_keys[c] == 'is_extended':
                    # for the 'is_extended' key, convert the boolean to a string; this is needed so that a fill string can be used for sources with missing images
                    is_extended_strings = [str(cat_table0[catalog_keys[c]].iloc[s]) for s in range(len(cat_table[catalog_keys[c]]))]
                    append_[f'{catalog_keys[c]}_{band}'] = is_extended_strings
                else:
                    append_[f'{catalog_keys[c]}_{band}'] = cat_table0[catalog_keys[c]]
    
        else:
            if band_catalogs[b] == 'MISSING':
                # if the skycell-matched image for the primary filter is missing, populate information subtable with missing key fill values
                print(f'MISSING BAND FILE for {band_catalogs[0]}')
                for c in range(len(catalog_keys)):
                    append_[f'{catalog_keys[c]}_{band}'] = num_src_in_image*[missing_key_fill_value[c]]
                    
            else:

                # open the catalog for the non-primary filter skycell-matched image
                parquet_file = pq.read_table(band_catalogs[b], filesystem=fs)
                columns = parquet_file.schema.names
                cat_table_match = parquet_file.to_pandas()

                # non-primary filter source locations converted into astropy coordinates
                RAs, DECs = cat_table_match['ra_centroid'], cat_table_match['dec_centroid']
                c1 = SkyCoord(RAs.to_numpy()*u.deg, DECs.to_numpy()*u.deg, frame='icrs')
                
                Cat_entries = []
                cat0_matches = []
                
                if len(c1)<1:
                    # if no sources are found in the existing sky-cell matched image, populate information subtable with missing key fillers
                    print(f'MISSING SOURCE INFO for band {band} match to {band_catalogs[0]}')
                    for c in range(len(catalog_keys)):
                        append_[f'{catalog_keys[c]}_{band}'] = num_src_in_image*[missing_key_fill_value[c]]
                        
                else:
                    # for every source location in the primary filter catalog, identify the closest matching source (within max_sep) from the non-primary image catalog
                    for j in range(len(c0)):
                        sep12 = c0[j].separation(c1).degree #calculate separation
                        coord_match = np.where((sep12 == min(sep12)) & (sep12 < max_sep.value))[0] #non-primary filter catalog index of source with min separation and sep < max_sep
                        if len(coord_match) > 0:
                            cat0_matches.append(j)
                            Cat_entries.extend(coord_match)
                            
                    # create a Matched_table that contains all the catalog information for the non-primary filter sources that match the indexing of the primary filter image/catalog
                    Matched_table = cat_table_match.iloc[Cat_entries].reset_index(drop=True)
                    Matched_table.index = cat0_matches #create indices matching the primary filter catalog information subtable so that two tables can be easily combined

                    # for each catalog_key, append the source information in Matched_table to the information subtable append_
                    for c in range(len(catalog_keys)):
                        
                        if catalog_keys[c] == 'is_extended':
                            is_extended_strings = [str(Matched_table[catalog_keys[c]].iloc[s]) for s in range(len(Matched_table[catalog_keys[c]]))]
                            is_extended_strings_series = pd.Series(is_extended_strings, index=cat0_matches)
                            append_[f'{catalog_keys[c]}_{band}'] = is_extended_strings_series
                        else:
                            append_[f'{catalog_keys[c]}_{band}'] = Matched_table[catalog_keys[c]]

    # append the skycell information for the current prog_match_file so that sources with missing information/files can be rechecked later
    append_['skycell'] = num_src_in_image*[skycell]
    
    # Add file names to tracking list to say the sources in the primary filter
    # images have been added to the full source catalog
    update_txt_file(band_files[0])

    
    num_band_files_per_image.append(len(band_files))  

    # append information subtable to the existing catalog
    catalog_df = pd.concat([catalog_df, append_], ignore_index=True)

            
duration = datetime.now()-startTime

In [ ]:
print(f'Number of images without sources = {ims_wo_sources}')
print(f'Number of band files per image = {np.unique(num_band_files_per_image)}')


In [ ]:
print('Time to create catalog:', duration)
print(catalog_df)

proc_files = get_uploaded_ids(filename)
print(proc_files)

In [ ]:
# Save the master catalog to disk
catalog_df.to_parquet('/teams/gz-roman/'+catalog_name+'.parquet')

# Checking for missing files

### Read in the existing master catalog

In [ ]:
cat_df = pd.read_parquet('/teams/gz-roman/'+catalog_name+'.parquet')

### Retrieve the band info for files on AWS

In [ ]:
band_info = {}
if file_source == 'soc_simulations':
    for i in trange(1,len(bands)):
    # for i in range(1):
        print(f'=========================================={bands[i]}==========================================')
        filter_plan_visit_level_info_array, filter_skycell_info_array, filter_prog_match_files, filter_visit_info = Retrieve_file_matches(dir_uri+'l3/', bands[i], All_files=False, in_plan=in_plan_, in_pass=in_pass_, in_segment=in_segment_)
        Print_file_info(filter_plan_visit_level_info_array, filter_skycell_info_array)
        band_info[bands[i]] = {'plan_visit_level_info': filter_plan_visit_level_info_array, 
                          'skycell_info_array':filter_skycell_info_array, 
                          'prog_match_files':filter_prog_match_files,
                          'visit_info':filter_visit_info}

### Check which master catalog rows are missing information and check the AWS for the missing files

In [ ]:

cat_df = pd.read_parquet('/teams/gz-roman/'+catalog_name+'.parquet')


num_updated_files = 0
for b in range(len(bands)):
    band = bands[b]
    print(f'band {band}')
    missing_i1 = np.where((cat_df[f'asdf_loc_{bands[b]}']=='MISSING'))[0]
    # missing_i1 = np.where((cat_df[f'asdf_loc_{bands[b]}']=='MISSING') | (cat_df[f'ra_centroid_{bands[b]}']==-999))[0]
    # print(missing_i1)
    # missing_entries_df = cat_df.iloc[missing_i1]
    print('Looping through missing catalog entries!')
    for i in missing_i1:
        # print(i)
        # print(cat_df.loc[i])
        # for each missing entry, convert source RA,DEC to astropy coord
        RA0, DEC0 = cat_df[f'ra_centroid_{bands[0]}'][i], cat_df[f'dec_centroid_{bands[0]}'][i]
        c0 = SkyCoord(RA0*u.deg, DEC0*u.deg, frame='icrs')

        # identify program file for the missing band info
        skycell_ = cat_df['skycell'][i]
        # print('skycell_ = ', skycell_)
        prog_match_files_i = np.where(band_info[band]['skycell_info_array'] == skycell_)[0] 
        # print('prog_match_files_i = ', prog_match_files_i)
        if len(prog_match_files_i) > 0: #if the matching skycell for the missing band exists
            prog_match_files = band_info[band]['prog_match_files'][prog_match_files_i][0]
            # print('prog_match_files = ', prog_match_files)
            
            # update the master catalog with the missing program_match_file
            cat_df.loc[i, f'asdf_loc_{band}'] = 's3://'+ prog_match_files
    
            # create catalog file name for the non-primary filter skycell-matched catalog
            file_name_cat = dir_uri+'l4/l3_source_catalog/'+prog_match_files.rsplit('/')[-1].rstrip('_coadd.asdf')+'_cat.parquet'
            # print('file_name_cat = ', file_name_cat)
    
            
            # open the catalog for the skycell-matched image
            parquet_file = pq.read_table(file_name_cat, filesystem=fs)
            columns = parquet_file.schema.names
            cat_table_match = parquet_file.to_pandas()
    
    
            # print('=================================================================================')
    
            # non-primary filter source locations converted into astropy coordinates
            RAs, DECs = cat_table_match['ra_centroid'], cat_table_match['dec_centroid']
            c1 = SkyCoord(RAs.to_numpy()*u.deg, DECs.to_numpy()*u.deg, frame='icrs')
            
            # Cat_entries = []
            # cat0_matches = []
            
            if len(c1)<1:
                # if no sources are found in the existing sky-cell matched image, populate information subtable with missing key fillers
                print(f'NO SOURCES FOUND IN SKYCELL-MATCHED BAND {band} FILE!!!')
                # for c in range(len(catalog_keys)):
                #     append_[f'{catalog_keys[c]}_{band}'] = num_src_in_image*[missing_key_fill_value[c]]
                    
            else:
                sep12 = c0.separation(c1).degree #calculate separation
                # print(sep12)
                coord_match = np.where((sep12 == min(sep12)) & (sep12 < max_sep.value))[0] #non-primary filter catalog index of source with min separation and sep < max_sep
                # print('coord_match', coord_match)
                if len(coord_match) > 0:
                    Matched_table = cat_table_match.iloc[coord_match].reset_index(drop=True)
                    Matched_table.index = [i]
                    # print('Matched_table = ', Matched_table)
                    
                    # for each catalog_key, append the source information in Matched_table to the existing catalog df
                    for c in range(len(catalog_keys)):
                        # print('catalog_keys[c] = ', catalog_keys[c])
                        # print('Matched_table[catalog_keys[c]] = ', Matched_table[catalog_keys[c]].values[0])
                        # print(cat_df[f'{catalog_keys[c]}_{band}'])
                        if catalog_keys[c] == 'is_extended':
                            is_extended_string = str(Matched_table[catalog_keys[c]].values[0])
                            # is_extended_strings_series = pd.Series(is_extended_strings, index=cat0_matches)
                            # append_[f'{catalog_keys[c]}_{band}'] = is_extended_strings_series
                            cat_df.loc[i, f'{catalog_keys[c]}_{band}'] = is_extended_string
                        else:
                            cat_df.loc[i, f'{catalog_keys[c]}_{band}'] = Matched_table[catalog_keys[c]].values[0]

            num_updated_files = num_updated_files + 1
            
print(cat_df)

In [ ]:
if num_updated_files >=1:
    print(f'New files were found! Creating updated master catalog: /teams/gz-roman/{catalog_name}_updated.parquet')
    cat_df.to_parquet('/teams/gz-roman/'+catalog_name+'_updated.parquet')

#### Check that all RA,DEC positions for the non-primary filters are within the max separation of the primary filter positions

In [ ]:
# print(catalog_df)
import numpy.ma as ma

RA_f106, DEC_f106 = cat_df['ra_centroid_f106'], cat_df['dec_centroid_f106']
skycoord_f106 = SkyCoord(RA_f106.to_numpy()*u.deg, DEC_f106.to_numpy()*u.deg, frame='icrs')

RA_f129, DEC_f129 = cat_df['ra_centroid_f129'], cat_df['dec_centroid_f129']
skycoord_f129 = []
diff_f106_f129 = []
for i in range(len(RA_f129)):
    if ((RA_f129[i] == float('nan')) or (RA_f129[i] == -999)):
        print('found NAN or -999 entry!')
        skycoord_f129.append(np.nan)
    else:
        indiv_skycoord_f129 = SkyCoord(RA_f129[i]*u.deg, DEC_f129[i]*u.deg, frame='icrs')
        skycoord_f129.append(indiv_skycoord_f129)
        diff_f106_f129.append(3600*skycoord_f106[i].separation(indiv_skycoord_f129).degree)

RA_f158, DEC_f158 = cat_df['ra_centroid_f158'], cat_df['dec_centroid_f158']
skycoord_f158 = []
diff_f106_f158 = []
for i in range(len(RA_f158)):
    if ((RA_f158[i] == float('nan')) or (RA_f158[i] == -999)):
        print('found NAN or -999 entry!')
        skycoord_f158.append(np.nan)
        diff_f106_f158.append(np.nan)
    else:
        indiv_skycoord_f158 = SkyCoord(RA_f158[i]*u.deg, DEC_f158[i]*u.deg, frame='icrs')
        skycoord_f158.append(indiv_skycoord_f158)
        diff_f106_f158.append(3600*skycoord_f106[i].separation(indiv_skycoord_f158).degree)


print(f'min,max positional differences = {min(diff_f106_f129)} - {max(diff_f106_f129)} arcsec')
print(f'min,max positional differences = {min(diff_f106_f158)} - {max(diff_f106_f158)} arcsec')


In [ ]:
# make sure there is only one entry per source id. At one point, the 
# above loop reset the src_id counter so this is mostly to test that 
# doesn't happen again. Eventually, we should remove this cell once 
# we are sure that issue does not pop up again.

cat_df = pd.read_parquet('/teams/gz-roman/'+catalog_name+'.parquet')

print(len(cat_df))
print(len(np.unique(cat_df['id_str'])))
# print(np.unique()

# Get the indices of the first occurrences
_, first_occurrence_indices = np.unique(cat_df['id_str'], return_index=True)

# Create a boolean mask of all False values
mask = np.zeros_like(cat_df['id_str'], dtype=bool)

# Mark the first occurrences as True
mask[first_occurrence_indices] = True

# Invert the mask to find all repeated/duplicate indices
repeated_indices = np.where(~mask)[0]

print("Repeated indices:", repeated_indices)
print(len(repeated_indices))
print(len(np.unique(repeated_indices)))